# 03 — LSTM Caption Decoder Training (Flickr8k)

This notebook follows **docs/updated_architecture.md** and continues from the first two notebooks:

- Image: **VGG16 (frozen) → 4096-dimensional saved features → Dense(256)**.
- Text: **clean captions → token IDs → padding → Embedding(256) → LSTM(256)**.
- Fusion: **Add → Dense(256) → Softmax(vocabulary) → next-word prediction**.

The LSTM is trained independently using saved image features. You **do not need to re-run VGG16 for each epoch**.

**Scope:** Training and validation only. Greedy decoding, beam search and BLEU metrics are later stages. The held-out test images are not used here.

## 0. Setup and project layout

Run the notebook from the repository root or `notebooks/` directory. Install packages with `pip install -r requirements.txt` in the environment first.

Required files:

```text
data/flickr8k/Images/          # Flickr8k photos
data/flickr8k/captions.txt
data/features/features.npy     # VGG16 fc2 outputs (4096 values each)
data/features/image_ids.json
data/features/progress.npy
src/data_preprocessing.py
src/feature_extraction.py
src/train_lstm.py
```

If `data/features/features.npy` is missing, first run `python -m src.feature_extraction` from the repository root. The saved image-feature files are excluded from Git and must exist on your computer/Colab runtime.

In [ ]:
from pathlib import Path
import sys
import json

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import tensorflow as tf
import keras
import matplotlib.pyplot as plt

from src.data_preprocessing import prepare_flickr8k
from src.feature_extraction import FEATURE_DIM, load_features
from src.train_lstm import build_caption_model, prepare_batches, plot_training_loss

print("Project root:", PROJECT_ROOT)
print("TensorFlow:", tf.__version__)
print("Visible GPUs:", tf.config.list_physical_devices("GPU"))

## 1. Training hyperparameters

The architecture sizes are fixed to **256 units** in `build_caption_model` to match your diagram. Adjust batch size, training epochs, learning rate and dropout in later experiments.

Use a different output folder for each experiment to avoid overwriting results.

In [ ]:
DATA_DIR = PROJECT_ROOT / "data" / "flickr8k"
FEATURES_DIR = PROJECT_ROOT / "data" / "features"
OUTPUT_DIR = PROJECT_ROOT / "models" / "lstm"

EPOCHS = 20
BATCH_SIZE = 128
LEARNING_RATE = 0.001
DROPOUT = 0.4
PATIENCE = 4
SEED = 42
MIN_COUNT = 1

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
keras.utils.set_random_seed(SEED)

## 2. Load and preprocess Flickr8k captions

Reuses your `data_preprocessing.py`. Images are split **80% training / 10% validation / 10% test** *by image*, ensuring that all five captions for an image stay in the same split.

The tokenizer and maximum caption length are fitted from **training captions only**, preventing leakage.

In [ ]:
prepared = prepare_flickr8k(data_dir=DATA_DIR, seed=SEED, min_count=MIN_COUNT)
prepared.save(OUTPUT_DIR)

print("Image splits:", {split: len(ids) for split, ids in prepared.splits.items()})
print("Vocabulary size:", prepared.vocab_size)
print("Maximum caption length:", prepared.max_length)
first_image = prepared.splits["train"][0]
print("Example cleaned caption:", prepared.captions[first_image][0])

## 3. Load saved VGG16 image features

`feature_extraction.py` has already resized and preprocessed images and run frozen ImageNet VGG16, saving the **fc2 layer's 4096-number vectors**.

This stage only loads those vectors. It does **not** retrain the CNN.

In [ ]:
if not (FEATURES_DIR / "features.npy").exists():
    raise FileNotFoundError(
        f"Missing {FEATURES_DIR / 'features.npy'}. Run: python -m src.feature_extraction"
    )

features, row_of = load_features(FEATURES_DIR, require_complete=True, mmap=True)
assert features.shape[1] == FEATURE_DIM
print("VGG16 features:", features.shape)
print("Example feature vector:", np.asarray(features[row_of[first_image]])[:8])

## 4. Create next-word training and validation batches

Each cleaned caption is expanded to many **(image vector, caption prefix) → next word** examples. For instance, `startseq a dog runs endseq` yields predictions for `a`, `dog`, `runs`, `endseq`.

`CaptionBatches` looks up the saved image vectors **only for the current batch**, avoiding huge repeated arrays. Targets remain integer IDs for sparse categorical cross-entropy.

In [ ]:
train_batches, n_train = prepare_batches(
    prepared, features, row_of, "train", batch_size=BATCH_SIZE, seed=SEED
)
val_batches, n_val = prepare_batches(
    prepared, features, row_of, "val", batch_size=BATCH_SIZE, seed=SEED
)

sample_inputs, sample_targets = train_batches[0]
print("Next-word training pairs:", n_train)
print("Next-word validation pairs:", n_val)
print("Image batch:", sample_inputs["image_features"].shape)
print("Caption batch:", sample_inputs["caption_input"].shape)
print("Target IDs:", sample_targets.shape)

## 5. Build the diagram's CNN–LSTM model

Two branches:

1. **Image:** saved 4096-feature VGG16 vector → Dropout → Dense(256, ReLU)
2. **Text:** padded token IDs → Embedding(256) → Dropout → LSTM(256)

**Feature fusion:** elementwise Add → Dense(256, ReLU) → Dropout → Softmax(vocabulary).

The softmax outputs a probability for every word in the training vocabulary. This is trained with **Adam + sparse categorical cross-entropy**. Captions are pre-padded and, by default, not masked: on a GPU, cuDNN only accepts masks for right-padded sequences, so a mask would stop training (see `mask_padding` in `build_caption_model`).

In [ ]:
model = build_caption_model(
    vocab_size=prepared.vocab_size,
    max_length=prepared.max_length,
    learning_rate=LEARNING_RATE,
    embedding_dim=256,
    lstm_units=256,
    dropout=DROPOUT,
)

model.summary()
print("Model input shapes:", model.input_shape)
print("Model output shape:", model.output_shape)

## 6. Train the LSTM decoder

- **EarlyStopping:** restores the model weights from the best validation epoch.
- **ModelCheckpoint:** saves the lowest-validation-loss model.
- **ReduceLROnPlateau:** lowers the learning rate if improvement stalls.
- **CSVLogger:** saves epoch-level results.

The test images are deliberately **excluded** during training and model selection.

In [ ]:
callbacks = [
    keras.callbacks.ModelCheckpoint(
        str(OUTPUT_DIR / "best_caption_model.keras"), monitor="val_loss",
        mode="min", save_best_only=True
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_loss", mode="min", patience=PATIENCE,
        restore_best_weights=True
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5,
        patience=max(2, PATIENCE // 2), min_lr=1e-6
    ),
    keras.callbacks.CSVLogger(str(OUTPUT_DIR / "training_history.csv")),
]

training = model.fit(
    train_batches,
    validation_data=val_batches,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1,
)

## 7. Save the trained model, experiment settings and plots

Store the exact tokenizer and splits beside the model to ensure the decoding code will map token indices back to the same words.

**Next-word accuracy is teacher-forced accuracy, not image-caption quality.** BLEU will be calculated after generating full captions on the held-out test images.

In [ ]:
model.save(OUTPUT_DIR / "caption_model.keras")
history = {key: [float(v) for v in values] for key, values in training.history.items()}
(OUTPUT_DIR / "history.json").write_text(json.dumps(history, indent=2), encoding="utf-8")
plot_training_loss(history, OUTPUT_DIR / "training_loss.png")

config = {
    "architecture": "VGG16(fc2 frozen) + Dense256 | Embedding256 + LSTM256 | Add + Dense256 + Softmax",
    "epochs_requested": EPOCHS,
    "epochs_run": len(history["loss"]),
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "dropout": DROPOUT,
    "patience": PATIENCE,
    "seed": SEED,
    "vocab_size": prepared.vocab_size,
    "max_length": prepared.max_length,
    "train_pairs": n_train,
    "val_pairs": n_val,
    "best_epoch": int(np.argmin(history["val_loss"]) + 1),
    "best_val_loss": float(min(history["val_loss"])),
}
(OUTPUT_DIR / "training_config.json").write_text(json.dumps(config, indent=2), encoding="utf-8")
print("Best epoch:", config["best_epoch"])
print("Best validation loss:", round(config["best_val_loss"], 4))
print("Saved to:", OUTPUT_DIR)

## 8. Visualise learning curves for your report

A large and increasing gap between training and validation loss can indicate overfitting. Use the plot to justify early stopping and any hyperparameter experiments.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(history["loss"], label="Training loss", marker="o", markersize=3)
ax.plot(history["val_loss"], label="Validation loss", marker="o", markersize=3)
ax.set_xlabel("Epoch")
ax.set_ylabel("Sparse categorical cross-entropy")
ax.set_title("LSTM training and validation loss")
ax.legend()
ax.grid(alpha=0.25)
plt.show()

## 9. Verify the checkpoint and plan evaluation

Loading the checkpoint confirms it is available for the **greedy decoding and beam search** implementation. A single softmax prediction is **not** a full caption.

**Next notebook (to implement):** generate captions using greedy and beam search (beam widths 3 and 5), then evaluate BLEU-1 through BLEU-4 against all five reference captions for each test image.

In [ ]:
best_model = keras.models.load_model(OUTPUT_DIR / "best_caption_model.keras")
print("Loaded:", best_model.name)
print("Inputs:", best_model.input_shape)
print("Output:", best_model.output_shape)
print("Test images reserved:", len(prepared.splits["test"]))